In [3]:
!pip install openpyxl
!pip install imblearn
!pip install opencv-python # Installs the OpenCV library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 19.1 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import RFE, mutual_info_classif

import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.models as models
import torch.optim as optim
from torchvision import transforms

from imblearn.over_sampling import SMOTE

import os
import warnings
warnings.filterwarnings("ignore")

In [5]:
# Define dataset path
pcos_extended_path = "/content/drive/MyDrive/pcos_detection/PCOS_extended_dataset.csv"  # Update with actual path

# Load the new dataset (PCOS Extended Data)
df_pcos_extended = pd.read_csv(pcos_extended_path)

# Display dataset information
print("✅ PCOS Extended Dataset Loaded!")
print("📊 Dataset Shape:", df_pcos_extended.shape)
print(df_pcos_extended.head())


✅ PCOS Extended Dataset Loaded!
📊 Dataset Shape: (2000, 44)
   Sl. No  Patient File No.  PCOS (Y/N)   Age (yrs)  Weight (Kg)  Height(Cm)   \
0     193               193           0          30    69.979147   167.708055   
1     360               360           0          36    63.711688   154.055877   
2      10                10           0          36    51.848631   149.059804   
3     278               278           1          29    66.893988   148.628036   
4      71                71           0          33    52.536198   150.767409   

         BMI  Blood Group  Pulse rate(bpm)   RR (breaths/min)  ...  \
0  23.185569           12                72                22  ...   
1  25.441392           13                70                18  ...   
2  23.928264           15                80                20  ...   
3  27.894935           15                72                18  ...   
4  23.079564           13                72                18  ...   

   Pimples(Y/N)  Fast food (Y/N)

In [6]:
# Check missing values before fixing
print("Missing Values Before Fixing:")
print(df_pcos_extended.isnull().sum())

# Handle missing values:
for col in df_pcos_extended.columns:
    if df_pcos_extended[col].dtype == "object":  # For categorical columns
        df_pcos_extended[col].fillna(df_pcos_extended[col].mode()[0], inplace=True)
    else:  # For numerical columns
        df_pcos_extended[col].fillna(df_pcos_extended[col].median(), inplace=True)

# Verify missing values are fixed
print("\n✅ Missing values handled successfully!")
print(df_pcos_extended.isnull().sum().sum(), "missing values remaining in PCOS Extended Data")


Missing Values Before Fixing:
Sl. No                    0
Patient File No.          0
PCOS (Y/N)                0
 Age (yrs)                0
Weight (Kg)               0
Height(Cm)                0
BMI                       0
Blood Group               0
Pulse rate(bpm)           0
RR (breaths/min)          0
Hb(g/dl)                  0
Cycle(R/I)                0
Cycle length(days)        0
Marraige Status (Yrs)     3
Pregnant(Y/N)             0
No. of abortions          0
  I   beta-HCG(mIU/mL)    0
II    beta-HCG(mIU/mL)    0
FSH(mIU/mL)               0
LH(mIU/mL)                0
FSH/LH                    0
Hip(inch)                 0
Waist(inch)               0
Waist:Hip Ratio           0
TSH (mIU/L)               0
AMH(ng/mL)                0
PRL(ng/mL)                0
Vit D3 (ng/mL)            0
PRG(ng/mL)                0
RBS(mg/dl)                0
Weight gain(Y/N)          0
hair growth(Y/N)          0
Skin darkening (Y/N)      0
Hair loss(Y/N)            0
Pimples(Y/N)      

In [7]:
# Select numerical columns (excluding target variable "PCOS (Y/N)")
numerical_cols = df_pcos_extended.select_dtypes(include=['number']).columns.tolist()
numerical_cols.remove("PCOS (Y/N)")

# Apply StandardScaler
scaler = StandardScaler()
df_pcos_extended[numerical_cols] = scaler.fit_transform(df_pcos_extended[numerical_cols])

print("✅ Feature Scaling completed (Z-score normalization applied).")


✅ Feature Scaling completed (Z-score normalization applied).


In [8]:
# Step 1: Standardize column names (Remove extra spaces & unwanted characters)
df_pcos_extended.columns = (
    df_pcos_extended.columns.str.replace("\s+", " ", regex=True)  # Replace multiple spaces with a single space
                              .str.strip()  # Remove leading and trailing spaces
                              .str.replace("I beta-HCG", "beta-HCG", regex=False)  # Fix incorrect column name
)

# Step 2: Remove non-informative columns (Sl. No, Patient File No.)
df_pcos_extended.drop(columns=["Sl. No", "Patient File No."], errors='ignore', inplace=True)

# Step 3: Convert all numerical values to float
for col in df_pcos_extended.columns:
    df_pcos_extended[col] = pd.to_numeric(df_pcos_extended[col], errors='coerce')

# Step 4: Fill NaN values with the median of each column
df_pcos_extended.fillna(df_pcos_extended.median(numeric_only=True), inplace=True)

# Step 5: Separate features (X) and target variable (y)
X = df_pcos_extended.drop(columns=["PCOS (Y/N)"])
y = df_pcos_extended["PCOS (Y/N)"]

# Step 6: Apply Recursive Feature Elimination (RFE) with RandomForestClassifier
estimator = RandomForestClassifier(random_state=42)
rfe = RFE(estimator, n_features_to_select=20)  # Select top 20 features
X_rfe = rfe.fit_transform(X, y)

# Step 7: Get selected feature names
selected_features = X.columns[rfe.support_]

# Convert back to DataFrame
X_selected = pd.DataFrame(X_rfe, columns=selected_features)

print("✅ Feature Selection completed successfully!")
print("Selected Features:", list(selected_features))


✅ Feature Selection completed successfully!
Selected Features: ['Weight (Kg)', 'Cycle(R/I)', 'Cycle length(days)', 'beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Fast food (Y/N)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)']


In [9]:
# Convert 'AMH(ng/mL)' to numeric in both DataFrames
df_pcos_extended['AMH(ng/mL)'] = pd.to_numeric(df_pcos_extended['AMH(ng/mL)'], errors='coerce')

In [15]:
# Use the new dataset directly
data = df_pcos_extended.copy()

# Display dataset information
print("✅ Extended PCOS Dataset Loaded!")
print("📊 Dataset Shape:", data.shape)
print(data.head())


✅ Extended PCOS Dataset Loaded!
📊 Dataset Shape: (2000, 42)
   PCOS (Y/N)  Age (yrs)  Weight (Kg)  Height(Cm)       BMI  Blood Group  \
0           0  -0.248511     0.913456    1.868618 -0.265934    -0.975716   
1           0   0.852718     0.365713   -0.390654  0.280117    -0.430014   
2           0   0.852718    -0.671054   -1.217445 -0.086155     0.661392   
3           1  -0.432049     0.643829   -1.288897  0.874030     0.661392   
4           0   0.302104    -0.610965   -0.934857 -0.291594    -0.430014   

   Pulse rate(bpm)  RR (breaths/min)  Hb(g/dl)  Cycle(R/I)  ...  Pimples(Y/N)  \
0        -0.302837          1.590653  0.975064    1.649007  ...      1.047108   
1        -0.790890         -0.707979 -0.753334   -0.602966  ...      1.047108   
2         1.649375          0.441337 -1.329466    1.649007  ...     -0.955011   
3        -0.302837         -0.707979  0.975064    1.649007  ...     -0.955011   
4        -0.302837         -0.707979 -1.099013   -0.602966  ...     -0.955011 

In [16]:

# Correct the column names to match those in the DataFrame
data = data.drop(columns=["  I   beta-HCG(mIU/mL)", "II    beta-HCG(mIU/mL)"], errors='ignore')
data = data.rename(columns={"PCOS (Y/N)": "Target"})


data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Target                 2000 non-null   int64  
 1   Age (yrs)              2000 non-null   float64
 2   Weight (Kg)            2000 non-null   float64
 3   Height(Cm)             2000 non-null   float64
 4   BMI                    2000 non-null   float64
 5   Blood Group            2000 non-null   float64
 6   Pulse rate(bpm)        2000 non-null   float64
 7   RR (breaths/min)       2000 non-null   float64
 8   Hb(g/dl)               2000 non-null   float64
 9   Cycle(R/I)             2000 non-null   float64
 10  Cycle length(days)     2000 non-null   float64
 11  Marraige Status (Yrs)  2000 non-null   float64
 12  Pregnant(Y/N)          2000 non-null   float64
 13  No. of abortions       2000 non-null   float64
 14  beta-HCG(mIU/mL)       2000 non-null   float64
 15  Ibet

In [19]:
columns_to_remove = [
    'Blood Group', 'Pulse rate(bpm)', 'RR (breaths/min)', 'Marraige Status (Yrs)',
    'Pregnant(Y/N)', 'No. of abortions', 'Ibeta-HCG(mIU/mL)', 'Hip(inch)',
    'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)',
    'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)',
    'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)',
    'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)',
    'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)',

]

# Remove the columns from the DataFrame
data = data.drop(columns=columns_to_remove)

# Show the cleaned data
data.head()

,Target,Age (yrs),Weight (Kg),Height(Cm),BMI,Hb(g/dl),Cycle(R/I),Cycle length(days),beta-HCG(mIU/mL),FSH(mIU/mL),LH(mIU/mL),FSH/LH
0,0,-0.248511,0.913456,1.868618,-0.265934,0.975064,1.649007,0.019336,-0.055998,-0.037035,-0.063036,-0.078227
1,0,0.852718,0.365713,-0.390654,0.280117,-0.753334,-0.602966,0.722456,-0.067180,-0.051622,-0.068581,-0.091416
2,0,0.852718,-0.671054,-1.217445,-0.086155,-1.329466,1.649007,-2.090025,-0.193512,-0.051878,-0.070261,-0.088208
3,1,-0.432049,0.643829,-1.288897,0.874030,0.975064,1.649007,0.019336,-0.193512,-0.053107,-0.062700,-0.102288
4,0,0.302104,-0.610965,-0.934857,-0.291594,-1.099013,-0.602966,0.019336,-0.192775,-0.041385,-0.076898,-0.001054


In [20]:
# Step 1: Check original class distribution
print("\n🔍 Class distribution before SMOTE:")
print(y.value_counts())

# Step 2: Apply SMOTE for perfect class balancing
smote = SMOTE(sampling_strategy='auto', random_state=42)  # Auto ensures full balancing
X_balanced, y_balanced = smote.fit_resample(X_selected, y)

# Step 3: Check new class distribution
print("\n✅ Class imbalance handled using SMOTE.")
print("New dataset shape:", X_balanced.shape)
print("Class distribution after SMOTE:\n", y_balanced.value_counts())



🔍 Class distribution before SMOTE:
PCOS (Y/N)
0    1392
1     608
Name: count, dtype: int64

✅ Class imbalance handled using SMOTE.
New dataset shape: (2784, 20)
Class distribution after SMOTE:
 PCOS (Y/N)
0    1392
1    1392
Name: count, dtype: int64


In [21]:
# 🔹 Step 1: Fix Data Size Mismatch
min_samples = min(len(X_selected), len(y_balanced))  # Ensure equal number of rows
X_selected = X_selected.iloc[:min_samples].reset_index(drop=True)
y_balanced = y_balanced.iloc[:min_samples].reset_index(drop=True)

print(f"✅ Data Size Fixed! Now X_selected: {X_selected.shape}, y_balanced: {y_balanced.shape}")

# 🔹 Step 2: Split Data into Training & Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_balanced, test_size=0.2, random_state=42)

print(f"✅ Train Size: {X_train.shape}, Test Size: {X_test.shape}")

# 🔹 Step 3: Convert Data to PyTorch Tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)  # Ensure 2D shape
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)  # Ensure 2D shape

# 🔹 Step 4: Create DataLoaders
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✅ Clinical Data Ready for MLP Model! 🚀")


✅ Data Size Fixed! Now X_selected: (2000, 20), y_balanced: (2000,)
✅ Train Size: (1600, 20), Test Size: (400, 20)
✅ Clinical Data Ready for MLP Model! 🚀


In [22]:
# 🔹 Step 1: Define the MLP Model
class PCOS_MLP(nn.Module):
    def __init__(self, input_dim):
        super(PCOS_MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 1),  # Output Layer
            nn.Sigmoid()  # Probability Output
        )

    def forward(self, x):
        return self.mlp(x)

# 🔹 Step 2: Define Model Parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]  # Number of clinical features

# Initialize Model
mlp_model = PCOS_MLP(input_dim).to(device)
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for probability output
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

print("✅ Clinical Data MLP Model Ready! 🚀")


✅ Clinical Data MLP Model Ready! 🚀


In [23]:
# 🔹 Training Function
def train_mlp_model(model, train_loader, criterion, optimizer, num_epochs=50):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            predicted = (outputs > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}, Accuracy: {accuracy:.2f}%")

# 🔹 Train the MLP Model
train_mlp_model(mlp_model, train_loader, criterion, optimizer, num_epochs=50)

print("✅ Clinical Data MLP Model Training Complete! 🎯")


Epoch [1/50], Loss: 24.8718, Accuracy: 75.81%
Epoch [2/50], Loss: 16.9859, Accuracy: 86.25%
Epoch [3/50], Loss: 13.8058, Accuracy: 89.44%
Epoch [4/50], Loss: 12.2055, Accuracy: 90.50%
Epoch [5/50], Loss: 12.2162, Accuracy: 90.44%
Epoch [6/50], Loss: 11.2408, Accuracy: 91.88%
Epoch [7/50], Loss: 10.8548, Accuracy: 92.06%
Epoch [8/50], Loss: 9.8982, Accuracy: 92.31%
Epoch [9/50], Loss: 9.5727, Accuracy: 92.25%
Epoch [10/50], Loss: 9.2578, Accuracy: 93.00%
Epoch [11/50], Loss: 8.6094, Accuracy: 93.38%
Epoch [12/50], Loss: 8.5918, Accuracy: 93.25%
Epoch [13/50], Loss: 8.0803, Accuracy: 93.56%
Epoch [14/50], Loss: 7.6560, Accuracy: 94.19%
Epoch [15/50], Loss: 8.1372, Accuracy: 93.81%
Epoch [16/50], Loss: 8.6089, Accuracy: 93.06%
Epoch [17/50], Loss: 7.8072, Accuracy: 94.25%
Epoch [18/50], Loss: 5.9006, Accuracy: 95.50%
Epoch [19/50], Loss: 6.3622, Accuracy: 95.06%
Epoch [20/50], Loss: 6.7420, Accuracy: 94.44%
Epoch [21/50], Loss: 6.7863, Accuracy: 94.62%
Epoch [22/50], Loss: 6.2739, Accurac

In [24]:
# 🔹 Evaluation Function
def evaluate_mlp_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            predicted = (outputs > 0.5).float()

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}% 🚀")

# 🔹 Evaluate the MLP Model
evaluate_mlp_model(mlp_model, test_loader)


Test Accuracy: 99.25% 🚀


In [25]:
import os
import torch

# Define the directory to save the model
save_directory = "/content/drive/MyDrive/pcos_detection/"
os.makedirs(save_directory, exist_ok=True)  # Creates the directory if it doesn't exist

# Define the model path
model_path = os.path.join(save_directory, "MLP_model2.pth")

# Save the model state_dict
torch.save(mlp_model.state_dict(), model_path)

print(f"Model saved at: {model_path}")

Model saved at: /content/drive/MyDrive/pcos_detection/MLP_model2.pth
